In [1]:
import os, sys
from typing import Any
from pandas import Series, DataFrame
from pandas import Timestamp, Timedelta
from pandas import read_pickle, concat
from pandas import Index, MultiIndex
from tqdm import tqdm

tests = dict[str, Any]()
for fn in os.listdir("../tests/"):
    if fn.endswith(".pickle"):
        key = fn[: 14]
        if key not in tests:
            tests[key] = list[str]()
        list.append(tests[key], fn)

print("Available tests:")
for key, files in tests.items():
    print(f" >> {key}: ({len(files)} files) ... {files}")
file_keys = ["ticks_p", "candles_p", "ticks_b", "candles_b"]

Available tests:
 >> 20260513023618: (4 files) ... ['20260513023618_candles_b.pickle', '20260513023618_candles_p.pickle', '20260513023618_ticks_b.pickle', '20260513023618_ticks_p.pickle']
 >> 20260513023619: (4 files) ... ['20260513023619_candles_b.pickle', '20260513023619_candles_p.pickle', '20260513023619_ticks_b.pickle', '20260513023619_ticks_p.pickle']
 >> 20260513023621: (4 files) ... ['20260513023621_candles_b.pickle', '20260513023621_candles_p.pickle', '20260513023621_ticks_b.pickle', '20260513023621_ticks_p.pickle']
 >> 20260513023622: (4 files) ... ['20260513023622_candles_b.pickle', '20260513023622_candles_p.pickle', '20260513023622_ticks_b.pickle', '20260513023622_ticks_p.pickle']
 >> 20260513023623: (4 files) ... ['20260513023623_candles_b.pickle', '20260513023623_candles_p.pickle', '20260513023623_ticks_b.pickle', '20260513023623_ticks_p.pickle']
 >> 20260513023625: (4 files) ... ['20260513023625_candles_b.pickle', '20260513023625_candles_p.pickle', '20260513023625_ticks_b

In [ ]:
side = "b"
columns = Index([*"ohlc"])
stats = dict()
ccount = dict()
iterator = tqdm(tests.items())
for key, files in iterator:
    ccount[key] = dict()
    iterator.set_description(f"Processing {key}...")
    dft = read_pickle(f"../tests/{key}_ticks_b.pickle")
    dfc = read_pickle(f"../tests/{key}_candles_b.pickle")
    dfp = read_pickle(f"../tests/{key}_candles_p.pickle")
    dft = dft.reset_index(["venue"], drop = True)
    dfc = dfc.reset_index(["venue"], drop = True)
    dfp = dfp.reset_index(["venue"], drop = True)
    dft = dft.groupby(["symbol", "time"])["p" + side].apply(list)
    dft = dft.reset_index("time")
    dft["time"] = dft["time"].dt.floor("1s")
    dft = dft.set_index("time", append = True)
    dfc = dfc[[*(columns + side), "volume"]]
    dfp = dfp[[*(columns + side), "volume"]]
    dfc.columns, dfp.columns = [*columns, "v"], [*columns, "v"]
    dfc = dfc.rename_axis("field", axis = "columns")
    dfp = dfp.rename_axis("field", axis = "columns")
    
    for tf in dfp.index.get_level_values("tf").unique():
        nc = dfc.query("tf == @tf").shape[0]
        np = dfp.query("tf == @tf").shape[0]
        ccount[key][tf] = {"b": nc, "p": np, "dt": Timedelta(
                tf[1 :] + tf[0].lower().replace("m", "min"))}
    dfc = dfc.xs("S1", level = "tf", drop_level = True)
    dfp = dfp.xs("S1", level = "tf", drop_level = True)

    comp = DataFrame.merge(dfc.stack("field").rename("b"), dfp.stack("field").rename("p"),
            left_index = True, right_index = True, how = "outer").sort_index().dropna()
    stat: Series = comp["b"] / comp["p"] - 1
    stat = stat.groupby(["symbol", "field"]).describe()
    stats[key] = stat

stats = concat(stats, names = ["test", "symbol", "field"])
ccount = DataFrame.from_dict(ccount, orient = "index")
ccount = ccount.stack().apply(Series)
ccount = ccount.rename_axis(["key", "tf"])
ccount["diff"] = ccount["b"] / ccount["p"] - 1
ccount["pat"] = ccount.apply("{0[b]} / {0[p]}".format, axis = "columns")

Processing 20260513023930...: 100%|██████████| 16/16 [00:44<00:00,  2.76s/it]


KeyError: 'b'

In [48]:
ccount["pat"].unstack("tf").sort_index().transpose()

key,20260513023618,20260513023619,20260513023621,20260513023622,20260513023623,20260513023625,20260513023626,20260513023628,20260513023629,20260513023650,20260513023709,20260513023725,20260513023737,20260513023821,20260513023859,20260513023930
tf,,,,,,,,,,,,,,,,
D1,0 / 1,0 / 1,0 / 1,0 / 1,0 / 2,0 / 2,0 / 2,0 / 2,0 / 3,0 / 2,0 / 1,0 / 1,0 / 6,0 / 4,0 / 2,0 / 2
H1,0 / 1,0 / 1,0 / 1,0 / 1,0 / 2,0 / 2,0 / 2,0 / 2,29 / 69,18 / 35,15 / 18,5 / 7,104 / 139,52 / 70,32 / 36,10 / 14
H12,0 / 1,0 / 1,0 / 1,0 / 1,0 / 2,0 / 2,0 / 2,0 / 2,2 / 6,1 / 3,0 / 2,0 / 1,4 / 12,0 / 6,0 / 4,0 / 2
H2,0 / 1,0 / 1,0 / 1,0 / 1,0 / 2,0 / 2,0 / 2,0 / 2,15 / 35,11 / 18,7 / 9,2 / 4,48 / 70,22 / 36,14 / 18,4 / 8
H3,0 / 1,0 / 1,0 / 1,0 / 1,0 / 2,0 / 2,0 / 2,0 / 2,10 / 23,7 / 12,4 / 6,1 / 3,36 / 47,14 / 24,8 / 12,2 / 6
H4,0 / 1,0 / 1,0 / 1,0 / 1,0 / 2,0 / 2,0 / 2,0 / 2,8 / 18,5 / 9,3 / 5,0 / 2,24 / 36,10 / 18,6 / 10,0 / 4
H6,0 / 1,0 / 1,0 / 1,0 / 1,0 / 2,0 / 2,0 / 2,0 / 2,5 / 12,4 / 6,1 / 3,0 / 2,14 / 24,2 / 12,2 / 6,0 / 4
H8,0 / 1,0 / 1,0 / 1,0 / 1,0 / 2,0 / 2,0 / 2,0 / 2,3 / 9,3 / 5,1 / 3,0 / 1,6 / 18,4 / 10,2 / 6,0 / 2
M1,17 / 42,12 / 22,7 / 11,2 / 4,40 / 80,26 / 42,18 / 21,6 / 9,1573 / 4092,1232 / 2059,817 / 1029,392 / 413,5354 / 8254,3484 / 4120,1980 / 2054,824 / 828
